In [ ]:
output_path = base_dir / "merged_repairability_dataset.csv"
merged_df.to_csv(output_path, index=False)
print("Saved merged dataset to:", output_path)
merged_df.head()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(merged_df["Repairability_Score"].dropna(), bins=15, kde=True)
plt.title("Repairability Score Distribution")
plt.xlabel("Repairability Score")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=merged_df, x="device_type", y="Repairability_Score")
plt.title("Repairability Score by Device Type")
plt.xticks(rotation=30)
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=summary, x="device_type", y="Repairability_Score", hue="label")
plt.title("Average Repairability Score by Device Type and Issue Label")
plt.xticks(rotation=30)
plt.show()

In [ ]:
summary = (
    merged_df.groupby(["device_type", "label"], dropna=False)["Repairability_Score"]
    .mean()
    .reset_index()
    .sort_values(["device_type", "label"])
)
print(summary)
summary

In [ ]:
# Create a simple device category key from the issue text
issue_df = issue_df.copy()
issue_df["device_type"] = issue_df["issue"].str.lower().str.extract(r"\b(phone|tablet|laptop|smartwatch)\b", expand=False).fillna("device")
issue_df["device_type"] = issue_df["device_type"].replace({
    "phone": "Smartphone",
    "tablet": "Tablet",
    "laptop": "Laptop",
    "smartwatch": "Smartwatch",
})

repairability_df["device_type"] = repairability_df["Category_Type"].str.lower().str.replace(" ", "", regex=False)
repairability_df["device_type"] = repairability_df["device_type"].replace({
    "smartphone": "Smartphone",
    "smartphones": "Smartphone",
    "laptop": "Laptop",
    "tablets": "Tablet",
    "tablet": "Tablet",
    "smartwatch": "Smartwatch",
})

merged_df = issue_df.merge(
    repairability_df[["device_type", "Manufacturer", "Repairability_Score"]].drop_duplicates(),
    on="device_type",
    how="left",
)

print("Merged rows:", merged_df.shape[0])
print("Missing repairability scores:", merged_df["Repairability_Score"].isna().sum())
merged_df.head()

In [ ]:
# Convert score columns to numeric and verify missing values
for frame_name, frame in [("repair_df", repair_df_clean), ("ifixit_df", ifixit_df_clean)]:
    print(f"\n{frame_name} missing score values:", frame["Repairability_Score"].isna().sum())
    print(f"{frame_name} non-null score count:", frame["Repairability_Score"].notna().sum())

repairability_df = pd.concat([repair_df_clean, ifixit_df_clean], ignore_index=True, sort=False)
repairability_df["Category_Type"] = repairability_df["Category_Type"].fillna("Unknown")
repairability_df.head()

In [ ]:
# Standardize repairability columns
repair_df_clean = repair_df.rename(columns={
    "Brand": "Manufacturer",
    "Model": "Product_Model",
    "Category": "Category_Type",
    "Vision Score": "Repairability_Score",
}).copy()
repair_df_clean = repair_df_clean[["Manufacturer", "Product_Model", "Category_Type", "Repairability_Score"]]

ifixit_df_clean = ifixit_df.rename(columns={
    "OEM": "Manufacturer",
    "Device": "Product_Model",
    "Score": "Repairability_Score",
    "Summary": "Summary_Description",
    "Scorecard Version": "Scorecard_Version",
}).copy()
ifixit_df_clean = ifixit_df_clean[["Manufacturer", "Product_Model", "Repairability_Score", "Summary_Description", "Scorecard_Version"]]

repair_df_clean["Manufacturer"] = repair_df_clean["Manufacturer"].astype(str).str.strip().str.title()
repair_df_clean["Category_Type"] = repair_df_clean["Category_Type"].astype(str).str.strip().str.title()
repair_df_clean["Repairability_Score"] = pd.to_numeric(repair_df_clean["Repairability_Score"], errors="coerce")

ifixit_df_clean["Manufacturer"] = ifixit_df_clean["Manufacturer"].astype(str).str.strip().str.title()
ifixit_df_clean["Repairability_Score"] = pd.to_numeric(ifixit_df_clean["Repairability_Score"], errors="coerce")

print("Repairability score range (repair_df):", repair_df_clean["Repairability_Score"].min(), "to", repair_df_clean["Repairability_Score"].max())
print("Repairability score range (iFixit):", ifixit_df_clean["Repairability_Score"].min(), "to", ifixit_df_clean["Repairability_Score"].max())
repair_df_clean.head()

In [ ]:
print("Issue dataset columns:")
print(issue_df.columns.tolist())
print("\nRepairability dataset columns:")
print(repair_df.columns.tolist())
print("\niFixit dataset columns:")
print(ifixit_df.columns.tolist())

print("\nIssue dataset info:")
issue_df.info()
print("\nRepairability dataset info:")
repair_df.info()
print("\niFixit dataset info:")
ifixit_df.info()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

base_dir = Path.cwd()
datasets_dir = base_dir.parent
all_csv_files = sorted(datasets_dir.glob("*.csv"))

all_datasets = {path.stem: pd.read_csv(path) for path in all_csv_files}

issue_df = all_datasets.get("device_issue_dataset_2000", pd.DataFrame())
repair_df = all_datasets.get("repairability-final", pd.DataFrame())
ifixit_df = all_datasets.get("iFixit Repairability Scores (Public) - Smartphones", pd.DataFrame())

print("Available datasets:", list(all_datasets.keys()))
for name, frame in all_datasets.items():
    print(f"\n{name}: shape={frame.shape}, columns={list(frame.columns)[:8]}")

print("\nissue_df shape:", issue_df.shape)
print("repair_df shape:", repair_df.shape)
print("ifixit_df shape:", ifixit_df.shape)
issue_df.head()

# Repairability Dataset Preprocessing
This notebook loads the gathered issue dataset and the repairability CSV files, then prepares a cleaned and merged analysis table for later modeling.